# 01b — Groundsource: Urban Analysis

**Purpose:** Characterise the urban composition of flood events, validate the 20% built-up
area threshold used throughout the pipeline, and understand the spatial and temporal
distribution of urban flood events.

**Input:**
- `outputs/groundsource_urban_df.parquet` — flood events with urban metrics (from 01a)

**Output:**
- Charts and printed summaries only. No files are saved.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

In [ ]:
# --- CONFIGURATION ---

INPUT_PATH          = r"D:\MY_CODES\global_urban_flood_database\groundsource\outputs\groundsource_urban_df.parquet"
URBAN_THRESHOLD_PCT = 20   # minimum built-up percentage to classify as an urban flood event

## 1. Load data

In [ ]:
df = pd.read_parquet(INPUT_PATH)
df['start_date'] = pd.to_datetime(df['start_date'])
df['end_date']   = pd.to_datetime(df['end_date'])
df['duration_days'] = (df['end_date'] - df['start_date']).dt.days + 1

print(f"Loaded {len(df):,} records")
print(df[['urban_percentage', 'urban_built_up_area_m2', 'polygon_total_area_m2']].describe())

## 2. Urban threshold — CDF analysis

In [ ]:
sorted_pct     = np.sort(df['urban_percentage'].values)
cdf            = np.arange(1, len(sorted_pct) + 1) / len(sorted_pct)
pct_below_thr  = (df['urban_percentage'] <= URBAN_THRESHOLD_PCT).mean() * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(sorted_pct, cdf * 100, color='steelblue', linewidth=1.5)
ax.axvline(URBAN_THRESHOLD_PCT, color='red', linestyle='--',
           label=f'Threshold = {URBAN_THRESHOLD_PCT}%')
ax.axhline(pct_below_thr, color='orange', linestyle=':',
           label=f'{pct_below_thr:.1f}% of events are below threshold')
ax.set_xlabel('Urban Built-up Percentage (%)')
ax.set_ylabel('Cumulative % of events')
ax.set_title('CDF of Urban Built-up Percentage — Groundsource')
ax.legend()
plt.tight_layout()
plt.show()

print(f"At the {URBAN_THRESHOLD_PCT}% threshold, {pct_below_thr:.1f}% of events are excluded.")

## 3. Filter for urban events and explore

In [ ]:
urban_df = df[
    (df['urban_percentage'] > URBAN_THRESHOLD_PCT) &
    (df['urban_percentage'] <= 100)
].copy()

print(f"Urban events (>{URBAN_THRESHOLD_PCT}%): {len(urban_df):,}  "
      f"({len(urban_df) / len(df) * 100:.1f}% of all events)")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Urban Flood Events (>{URBAN_THRESHOLD_PCT}% built-up) — Groundsource', fontsize=13)

# Panel 1: Area distribution
log_area = np.log10(urban_df['area_km2'].clip(lower=1e-4))
axes[0, 0].hist(log_area, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0, 0].axvline(np.log10(urban_df['area_km2'].median()), color='red', linestyle='--',
                   label=f"Median = {urban_df['area_km2'].median():.2f} km²")
axes[0, 0].set_xlabel('log₁₀(Area [km²])')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Flood Area Distribution')
axes[0, 0].legend()

# Panel 2: Duration distribution
dur = urban_df['duration_days'].value_counts().sort_index()
axes[0, 1].bar(dur.index, dur.values, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0, 1].set_xlabel('Duration (days)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Duration Distribution')
axes[0, 1].set_xticks(dur.index)

# Panel 3: Annual distribution
year_counts = urban_df.groupby(urban_df['start_date'].dt.year).size()
axes[1, 0].bar(year_counts.index, year_counts.values, color='steelblue', edgecolor='white', linewidth=0.5)
axes[1, 0].set_xlabel('Year')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Annual Distribution')
axes[1, 0].tick_params(axis='x', rotation=45)

# Panel 4: Urban percentage histogram (log y)
axes[1, 1].hist(urban_df['urban_percentage'], bins=50,
                color='steelblue', edgecolor='white', linewidth=0.3, log=True)
axes[1, 1].axvline(urban_df['urban_percentage'].median(), color='red', linestyle='--',
                   label=f"Median = {urban_df['urban_percentage'].median():.1f}%")
axes[1, 1].set_xlabel('Urban Built-up Percentage (%)')
axes[1, 1].set_ylabel('Count (log scale)')
axes[1, 1].set_title(f'Urban Percentage Distribution (>{URBAN_THRESHOLD_PCT}%)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 4. Urban percentage vs flood area scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(
    urban_df['area_km2'].clip(lower=1e-3),
    urban_df['urban_percentage'],
    s=2, alpha=0.3, color='steelblue'
)
ax.set_xscale('log')
ax.set_xlabel('Flood Area (km², log scale)')
ax.set_ylabel('Urban Built-up Percentage (%)')
ax.set_title('Urban Percentage vs Flood Area — Groundsource Urban Events')
ax.axhline(URBAN_THRESHOLD_PCT, color='red', linestyle='--',
           label=f'Threshold = {URBAN_THRESHOLD_PCT}%')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Summary

In [ ]:
print("=" * 55)
print(f"  URBAN ANALYSIS SUMMARY (>{URBAN_THRESHOLD_PCT}% threshold)")
print("=" * 55)
print(f"  All events             : {len(df):>10,}")
print(f"  Urban events           : {len(urban_df):>10,}  ({len(urban_df)/len(df)*100:.1f}%)")
print(f"  Median urban %         : {urban_df['urban_percentage'].median():>10.1f}%")
print(f"  Median flood area      : {urban_df['area_km2'].median():>10.3f} km²")
print(f"  Year range (urban)     : {urban_df['start_date'].dt.year.min()} – {urban_df['start_date'].dt.year.max()}")
print("=" * 55)